In [1]:
"""******************************************************************************************
* HumARConoid-Sim2Real
*
* Sim2Real Transfer to Optimize Humanoid Locomotion Strategy using Reinforcement Learning
*
*     https://github.com/S-CHOI-S/HumARConoid-Sim2Real.git
*
* Advanced Robot Control Lab. (ARC)
* 	  @ Korea Institute of Science and Technology
*
*	  https://sites.google.com/view/kist-arc
*
******************************************************************************************"""

"* Authors: Sol Choi (Jennifer) *"

import torch
import onnx
import onnxruntime as ort
import numpy as np

from policy import ARC_G1_POLICY_DIR

In [ ]:
import onnxruntime
print(onnxruntime.__version__)

# 1. Load the PyTorch model (.pt)
# pytorch_model = torch.jit.load('./policy/g1/policy_korea.pt')
pytorch_model = torch.jit.load('../policy/sol/0803/policy_straight.pt')
print(pytorch_model.eval())

# 2. Load the ONNX model (.onnx)
onnx_model = onnx.load('../policy/sol/0803/policy_straight.onnx')
onnx.checker.check_model(onnx_model)  # Check if the ONNX model is valid

# Add hidden states (h_in, c_in) if needed for LSTM
h_in = np.zeros((1, 1, 64), dtype=np.float32)  # For LSTM: (num_layers, batch_size, hidden_size)
c_in = np.zeros((1, 1, 64), dtype=np.float32)

# 3. Create a random input tensor (for testing, match input size of your model)
# Make sure this matches the input shape that both models expect
# input = np.array([
#     0, 0.000872665, -0.00174533, 0.0525013, -0.00172486, -0.99862, 0, -0, -0,
#     -0.00176501, -0.000971521, 0.000468716, -0.00059253, -0.0104093, 0.0032868,
#     -0.00184537, 0.00151694, -0.000374973, -0.000967503, -0.0140385, -0.00383185,
#     -0.000257124, -0.000490874, -0.000257124, 5.45415e-05, 2.67367e-05,
#     -0.000136815, -0.0031712, -0.000381791, 0.000171416, -0.00163625,
#     -5.39856e-05, -0.000182675, -0.571938, 0.761322, -0.507879, 1.43071,
#     -0.50739, 0.876752, 0.186851, -3.43357, 0.22636, 1.31718, 4.00805,
#     0.0507264, 0.95106, -0.309007
# ], dtype=np.float32)

for _ in range(10):
    input = np.random.randn(47).astype(np.float32)
    
    # input = np.zeros(98).astyp?e(np.float32)
    # input = np.array([0.7792388768086855, -0.33621711815312194, 0.9635528585084845, -0.044070512841866805, -0.6130485225489184, -0.6733710942890032, -0.5584583103594003, -0.689277848953121, -0.6095517614792743, -0.6215307107879224, 0.7457869485548547, 0.5302465349723864, 0.4534002939123658, 0.6025710524537691, 0.4906513804169097, -0.2207418165594437, -0.18080561447591958, -0.9383707224437163, 0.19011532903596673, -0.475924566560874, -0.2901719887699563, 0.8671404337535262, -0.2718975405421542, -0.8389477310119327, 0.8282952771046359, 0.16662376490483566, 0.44988667161631124, -0.49405911044935147, 0.5863696512827363, -0.8577171911215211, 0.17389028436792442, 0.3611538728188841, -0.5088410772783492, 0.5304233155189562, -0.25056500508785784, 0.902463516168351, -0.6718349359524858, -0.6894858065923102, 0.7760530516797042, -0.511283457529532, 0.9494247311180728, -0.707324884904424, -0.5134282930687408, -0.6734094593747422, 0.79202904102541, -0.6344084871236664, 0.9589739639144268, -0.29044265042638107, -0.6262988613741696, 0.6654942186547115, 0.9222269032325623, 0.2685580694078473, 0.28843074225871446, -0.1786166686938666, 0.9747603653221537, -0.8012985972890889, -0.9898738066889177, -0.44732210652246596, 0.5260777813685371, 0.4386738821571763, 0.24358863709297718, -0.6169254380694591, 0.41984478926479696, -0.1310820864812472, 0.2817885597400118, 0.3923056475768836, 0.8552977837294935, -0.02101031796665742, 0.4349040836598652, 0.7295985637495486, 0.9636330701783162, -0.051977563608943456, 0.48665698890161324, -0.6759424514252859, 0.8980359229848096, 0.8176299055556828, -0.9926408579807076, -0.34492726646753513, 0.6408671365286223, -0.05360094890584666, -0.2063874502830374, 0.1615374704456154, -0.7339847703395384, -0.20912532127646966, 0.44402961848889966, 0.17415975253181792, -0.970762005290609, -0.3115218482056288, -0.7131522597698314, -0.2086256234050654, 0.7724400741033792, 0.6491114819179202, -0.00930659793956834, -0.3012121647180399, -0.808656479460955, 0.8066276221844542, 0.9804120012065449, -0.2394455708519554
    # ], dtype=np.float32)
    input_tensor = torch.from_numpy(input).unsqueeze(0) # Example input shape, modify as needed

    # 4. Get the output from the PyTorch model
    with torch.no_grad():
        pytorch_output = pytorch_model(input_tensor)

    # 5. Get the output from the ONNX model using onnxruntime
    onnx_session = ort.InferenceSession('../policy/sol/0803/policy_straight.onnx', providers=['CPUExecutionProvider'])

    # Ensure the correct input shape for ONNX model
    # ONNX expects numpy input
    onnx_input = {onnx_session.get_inputs()[0].name: input_tensor.numpy()}

    # Update the input dictionary to include h_in, c_in
    # onnx_input[onnx_session.get_inputs()[1].name] = h_in
    # onnx_input[onnx_session.get_inputs()[2].name] = c_in

    # print(onnx_session.get_outputs()[0].name)
    # print(onnx_session.get_outputs()[1].name)
    # print(onnx_session.get_outputs()[2].name)

    # Run the ONNX model with the inputs
    # onnx_output, h_in, c_in = onnx_session.run(None, onnx_input)
    onnx_output = onnx_session.run(None, onnx_input)
    # print(onnx_output)

    # 6. Compare the outputs (PyTorch vs ONNX)
    # Convert ONNX output to a PyTorch tensor
    onnx_output_tensor = torch.tensor(onnx_output)

    # 7. Check if the outputs are close
    # Note: Due to possible floating point precision differences, we use torch.allclose
    are_outputs_close = torch.allclose(pytorch_output, onnx_output_tensor, atol=1e-5)  # Tolerance can be adjusted

    # 8. Print the result
    if are_outputs_close:
        print("The outputs are the same!")
    else:
        print("The outputs are different.")
        
    print(f"PyTorch output:\n  {pytorch_output}")
    print(f"ONNX output:\n  {onnx_output_tensor}")
    print("="*50)
    
# for tensor in onnx_model.graph.initializer:
#     print(f"Name: {tensor.name}")
#     print(f"Shape: {tensor.dims}")
#     print(f"Data: {tensor.float_data}")
#     print("="*50)

1.11.0
RecursiveScriptModule(
  original_name=_TorchPolicyExporter
  (actor): RecursiveScriptModule(
    original_name=Sequential
    (0): RecursiveScriptModule(original_name=Linear)
    (1): RecursiveScriptModule(original_name=ELU)
    (2): RecursiveScriptModule(original_name=Linear)
    (3): RecursiveScriptModule(original_name=ELU)
    (4): RecursiveScriptModule(original_name=Linear)
    (5): RecursiveScriptModule(original_name=ELU)
    (6): RecursiveScriptModule(original_name=Linear)
  )
  (normalizer): RecursiveScriptModule(original_name=Identity)
)
The outputs are the same!
PyTorch output:
  tensor([[ 0.8067, -1.8320, -1.3010, -0.7751,  0.1674,  0.2508,  0.5178,  0.4571,
         -0.4433,  0.3203, -1.7337, -0.8035]])
ONNX output:
  tensor([[[ 0.8067, -1.8320, -1.3010, -0.7751,  0.1674,  0.2508,  0.5178,
           0.4571, -0.4433,  0.3203, -1.7337, -0.8035]]])
The outputs are the same!
PyTorch output:
  tensor([[ 1.8382, -0.7015,  0.5916, -1.3312,  0.1057,  0.0270,  0.7670, -0.600

/tmp/ipykernel_10035/1465862939.py:64: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:245.)
  onnx_output_tensor = torch.tensor(onnx_output)
